In [1]:
# %%
import torch
import pandas as pd
from torch_geometric.data import Data
from pathlib import Path
import os

os.chdir(Path().cwd().parent)
from modelling import get_dataframes
from modelling.metrics.metricstracker import MetricsTracker
import datetime
from graph_modelling.utils.load_data import (
    load_train_val_data,
    load_test_data,
    read_csv_files,
)
from graph_modelling.utils.tune_gnn import objective
from graph_modelling.models.temporalgnn import TemporalGNN
from graph_modelling.models.basicgnn import BasicGNN
from graph_modelling.models.attentiongnn import AttentionGNN
from graph_modelling.models.temporalattentiongnn import GATGRUGNN
import pickle
import optuna
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt
from graph_modelling.utils.test_gnn import predict_and_evaluate
from graph_modelling.utils.train_gnn import train
import numpy as np
import argparse

HABROK = bool(0)  # set to True if using HABROK; it will print
# all stdout to a .txt file to log progress
BASE_DIR = Path.cwd()
MODEL_PATH = BASE_DIR / "results" / "gnn_results" / "models"
DATA_DIR = BASE_DIR / "data" / "data_combined"
ALL_DIR = DATA_DIR / "all"

print("BASE_DIR: ", BASE_DIR)
print("MODEL_PATH: ", MODEL_PATH)
print("ALL_DIR: ", ALL_DIR)

torch.manual_seed(34)  # set seed for reproducibility

N_HOURS_U = 72  # number of hours to use for input
N_HOURS_Y = 24  # number of hours to predict
N_HOURS_STEP = 24  # "sampling rate" in hours of the data; e.g. 24

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")



Running __init__.py for data pipeline...
Modelling package initialized

/opt/rocm/lib/libamd_smi.so: cannot open shared object file: No such file or directory
Unable to find amdsmi library try installing amd-smi-lib from your package manager


2025-04-16 20:33:02.675147: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-16 20:33:02.675206: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-16 20:33:02.675235: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-16 20:33:02.683254: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-16 20:33:03.491811: W tensorflow/compiler/

/home/nick/bachelor-project/forecasting_smog_DL_GNN
BASE_DIR:  /home/nick/bachelor-project/forecasting_smog_DL_GNN
MODEL_PATH:  /home/nick/bachelor-project/forecasting_smog_DL_GNN/results/gnn_results/models
ALL_DIR:  /home/nick/bachelor-project/forecasting_smog_DL_GNN/data/data_combined/all
cuda


/home/nick/bachelor-project/forecasting_smog_DL_GNN/.venv/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = TemporalGNN(
    input_dim=7,
    output_dim=24,
    hidden_dim=32,
    gcn_layers=3,
    rnn_layers=1,
    rnn_dropout=0.1,)

with open(ALL_DIR / "geometric_pkl" / "test_dataset.pkl", "rb") as f:
    test_dataset = pickle.load(f)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
)



In [3]:
model.load_state_dict(
    torch.load(
        MODEL_PATH / "final_model_temporalgnn_20250416-201604.pt",
        map_location=device,
    )
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001394144770523156,
    weight_decay=0.00042097044805796443,
)

In [4]:
model.to(device)

TemporalGNN(
  (convs): ModuleList(
    (0): GCNConv(7, 32)
    (1-2): 2 x GCNConv(32, 32)
  )
  (rnn): GRU(32, 32, batch_first=True)
  (fc_out): Linear(in_features=32, out_features=24, bias=True)
)

In [5]:
# Load y_min and y_max
with open(ALL_DIR / "geometric_pkl" / "y_min_max.pkl", "rb") as f:
    y_min, y_max = pickle.load(f)

In [6]:
y_min

array([[[ 1.4 ],
        [-1.  ],
        [-0.28]]], dtype=float32)

In [7]:
y_max

array([[[143.4 ],
        [156.4 ],
        [ 95.46]]], dtype=float32)

In [8]:
global_rmse, rmse_per_node, mean_no2_rmse = predict_and_evaluate(
    model, test_loader, device, 24, y_min, y_max, N_HOURS_Y
)

Node 0: NO2 RMSE = 19.2027
Node 1: NO2 RMSE = 21.2779
Node 2: NO2 RMSE = 12.9630

🌍 Global RMSE (all nodes, all time steps): 18.1616
📉 Mean NO2 RMSE across nodes: 17.8145
